# 05. Resource-Adaptive Federated KAN with PPO Meta-Controller
## Master Thesis: Resource-Adaptive Federated Kolmogorov-Arnold Networks

### Mathematical Formulation of Closed-Loop Adaptive FL
In this notebook, we analyze the Reinforcement Learning meta-controller designed to solve the multi-objective optimization problem:
$$\\max_{\\pi} \\mathbb{E} \\left[ \\sum_{t=0}^T R_t \\right]$$
where the scalarized multi-objective step reward is defined as:
$$R_t = \\Delta \\text{Acc}_t - \\lambda_1 \\sum_{i=1}^N |\\theta_i^t| - \\lambda_2 \\sum_{i=1}^N \\Delta \\tau_i^t$$
- **Telemetry State Vector**:
  $$S_i^t = [L_{\\text{local}}^i, \\text{Var}(x^i), \\Delta \\tau^i, \\text{bw}^i, \\kappa^i] \\in [0, 1]^5$$
- **Multi-Discrete Action Space**: Each client selects from $12$ grid configurations:
  $$(G_i^t, k_i^t) \\in \\{3, 5, 7, 10\\} \\times \\{2, 3, 4\\}$$


In [ ]:
import os
import sys
from pathlib import Path

# Set project root
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json
from IPython.display import Markdown, display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
print(f"Working directory: {PROJECT_ROOT}")


---
### 1. Load Experimental Results Across Adaptive Controllers
We compare the deterministic **HeuristicController** (quantile-based complexity score) against the trained **PPOController**.


In [ ]:
ppo_summary_csv = PROJECT_ROOT / "results/adaptive_kan/ppo_master_summary.csv"
heuristic_summary_csv = PROJECT_ROOT / "results/adaptive_kan/heuristic_master_summary.csv"

df_ppo = pd.read_csv(ppo_summary_csv)
df_heuristic = pd.read_csv(heuristic_summary_csv)

print("PPO Meta-Controller Results:")
display(df_ppo[["alpha", "final_accuracy", "final_macro_f1", "mean_parameters_per_round", "mean_controller_overhead_ms", "total_reward"]])

print("Heuristic Controller Results:")
display(df_heuristic[["alpha", "final_accuracy", "final_macro_f1", "mean_parameters_per_round", "total_reward"]])


---
### 2. Thesis Master Benchmark (Table 4.1)
Below is the master benchmark comparing all 5 architectures across non-IID skews $\alpha \in \{0.1, 0.5, 1.0\}$ on the 18,175-heartbeat test split.


In [ ]:
table_path = PROJECT_ROOT / "results/ablations/table_4_1_benchmark.md"
if table_path.exists():
    with open(table_path) as f:
        md_content = f.read()
    display(Markdown(md_content))
else:
    print("Table 4.1 file not found.")


---
### 3. Non-IID Robustness & Degradation Rate Analysis
Evaluating how severely performance degrades when shifting from uniform-like ($\alpha=1.0$) to severely skewed ($\alpha=0.1$) distributions.


In [ ]:
robustness_csv = PROJECT_ROOT / "results/ablations/non_iid_robustness.csv"
if robustness_csv.exists():
    df_rob = pd.read_csv(robustness_csv)
    display(df_rob)


---
### 4. Thesis Deliverable Figures (Chapter 4)
We display the generated publication-quality figures:
- **Figure 4.1**: Pareto Frontier of Global Accuracy vs. Communication Cost
- **Figure 4.2**: 1D B-Spline Interpretability Comparison across Heterogeneous Edge Profiles
- **Figure 4.3**: Minority Class F1 Retention under Non-IID Skew
- **Figure 4.4**: Controller Overhead & Wall-Clock Latency Breakdown


In [ ]:
import matplotlib.image as mpimg

fig, axes = plt.subplots(2, 2, figsize=(18, 14))

# Figure 4.1
img1 = mpimg.imread(PROJECT_ROOT / "results/figures/fig4_1_pareto_frontier.png")
axes[0, 0].imshow(img1)
axes[0, 0].axis("off")
axes[0, 0].set_title("Figure 4.1: Pareto Frontier", fontsize=12, fontweight="bold")

# Figure 4.2
img2 = mpimg.imread(PROJECT_ROOT / "results/figures/fig4_2_spline_interpretability.png")
axes[0, 1].imshow(img2)
axes[0, 1].axis("off")
axes[0, 1].set_title("Figure 4.2: Spline Interpretability", fontsize=12, fontweight="bold")

# Figure 4.3
img3 = mpimg.imread(PROJECT_ROOT / "results/figures/fig4_3_classwise_f1.png")
axes[1, 0].imshow(img3)
axes[1, 0].axis("off")
axes[1, 0].set_title("Figure 4.3: Minority Class F1 Preservation", fontsize=12, fontweight="bold")

# Figure 4.4
img4 = mpimg.imread(PROJECT_ROOT / "results/figures/fig4_4_controller_overhead.png")
axes[1, 1].imshow(img4)
axes[1, 1].axis("off")
axes[1, 1].set_title("Figure 4.4: Computational Overhead Breakdown", fontsize=12, fontweight="bold")

plt.tight_layout()
plt.show()


---
### 5. Multi-Objective Reward Weight Sensitivity Sweep ($\lambda_1$ vs $\lambda_2$)
Heatmap analysis demonstrating the sensitivity of the policy to the parameter penalty coefficient $\lambda_1$ and the latency penalty coefficient $\lambda_2$.


In [ ]:
sensitivity_csv = PROJECT_ROOT / "results/ablations/reward_sensitivity.csv"
if sensitivity_csv.exists():
    df_sens = pd.read_csv(sensitivity_csv)
    print("Reward Sensitivity Grid (9 Parameter Pairs):")
    display(df_sens)

    # Display generated heatmap
    img_sens = mpimg.imread(PROJECT_ROOT / "results/figures/reward_sensitivity_heatmap.png")
    plt.figure(figsize=(14, 6))
    plt.imshow(img_sens)
    plt.axis("off")
    plt.title("Reward Sensitivity Sweep (λ1 vs. λ2)", fontsize=13, fontweight="bold")
    plt.show()


---
### 6. Controller Overhead Quantification (Section 3.9.4)
- **RL Decision Overhead**: Mean PPO forward pass inference latency: **$1.19\text{ ms}$** across 8 clients.
- **Local Edge Training Latency**: **$4,080\text{ ms}$** per communication round.
- **Overhead Percentage**:
  $$\\frac{1.19\\text{ ms}}{4080.0\\text{ ms}} \\times 100\\% = 0.029\\%$$
- **RAM Footprint**: **$1.25\text{ MB}$** controller resident memory.
- **Conclusion**: The reinforcement learning closed-loop meta-controller operates in real time with negligible edge and server computational overhead.


---
### 7. Performance vs. Communication Cost Benchmark (Figures C1–C8 & Table)
Comprehensive evaluation comparing all 8 methods across non-IID Dirichlet skews ($lpha \in \{0.1, 0.5, 1.0\}$) over 5 random seeds (42–46).
- **Fig C1**: Accuracy vs. Cumulative Communication (Convergence Curves)
- **Fig C2**: Macro F1 & Balanced Accuracy vs. Cumulative Communication
- **Fig C3**: Final Performance vs. Total Communication (Empirical Pareto Frontiers)
- **Fig C4**: Per-Method Small Multiples
- **Fig C5**: Communication Needed to Reach Target Accuracy Thresholds
- **Fig C6**: Communication Volume & Client Config Histograms
- **Fig C7**: Test Accuracy vs. Cumulative Simulated Transmission Time
- **Fig C8**: Relative Parameter Communication Savings Summary


In [ ]:
from IPython.display import display, Markdown
import matplotlib.image as mpimg

# 1. Display Markdown benchmark table
comm_table_path = PROJECT_ROOT / "results/ablations/comm_vs_performance.md"
if comm_table_path.exists():
    display(Markdown(comm_table_path.read_text()))

# 2. Display Key Communication Figures (Fig C1 & Fig C3)
fig_dir = PROJECT_ROOT / "results/figures/comm"
fig_c1 = fig_dir / "fig_c1_acc_vs_cum_comm.png"
fig_c3 = fig_dir / "fig_c3_pareto_scatter.png"

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 18))
if fig_c1.exists():
    ax1.imshow(mpimg.imread(str(fig_c1)))
    ax1.axis("off")
    ax1.set_title("Figure C1: Test Accuracy vs. Cumulative Communication (Convergence Curves)", fontsize=14, fontweight="bold")

if fig_c3.exists():
    ax2.imshow(mpimg.imread(str(fig_c3)))
    ax2.axis("off")
    ax2.set_title("Figure C3: Final Performance vs. Total Communication (Pareto Frontiers)", fontsize=14, fontweight="bold")

plt.tight_layout()
plt.show()
